In [ ]:
# DB_CONNECTION=mysql
# DB_HOST=127.0.0.1
# DB_PORT=3306
# DB_DATABASE=bloodbridge_db
# DB_USERNAME=root
# DB_PASSWORD=password

In [1]:
from sqlalchemy import create_engine, text

DB_USERNAME = "root"
DB_PASSWORD = "password"         
DB_HOST     = "127.0.0.1"
DB_PORT     = "3306"
DB_DATABASE = "bloodbridge_db"

engine = create_engine(
    f"mysql+pymysql://{DB_USERNAME}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_DATABASE}"
)

# تحقق من الاتصال
with engine.connect() as conn:
    result = conn.execute(text("SELECT 1"))
    print("✅ Connected to MySQL successfully!")

✅ Connected to MySQL successfully!


In [2]:
with engine.connect() as conn:
    result = conn.execute(text("SHOW TABLES"))
    tables = [row[0] for row in result]

print("الجداول في قاعدة البيانات:")
for table in tables:
    print(f"  → {table}")

الجداول في قاعدة البيانات:
  → achievements
  → announcements
  → appointments
  → blood_requests
  → cache
  → cache_locks
  → contact_messages
  → donor_achievements
  → donor_behavioral_metrics
  → donor_health_profiles
  → donor_predictive_scores
  → donors
  → eligibility_logs
  → failed_jobs
  → governorates
  → job_batches
  → jobs
  → migrations
  → model_training_logs
  → notifications
  → organizations
  → password_reset_tokens
  → request_responses
  → sessions
  → settings
  → users


In [4]:
import pandas as pd

with engine.connect() as conn:
    df_donors = pd.read_sql(text("""
        SELECT
            d.id,
            dhp.blood_type,
            d.governorate_id,
            d.lat,
            d.lng,
            u.name,
            u.email
        FROM donors d
        JOIN users u ON d.user_id = u.id
        Join donor_health_profiles dhp on d.id = dhp.donor_id
        LIMIT 10
    """), conn)

print(f"عدد المتبرعين: {len(df_donors)}")
print(df_donors)

عدد المتبرعين: 10
   id  blood_type  governorate_id      lat     lng                 name  \
0   1           1               1  31.5015  34.466    محمد خالد أبو عمر   
1   2           3               1      NaN     NaN      أحمد يوسف الحسن   
2   3           5               1  31.5100  34.461      عمر سعيد الشريف   
3   4           7               1  31.5190  34.447  خالد إبراهيم النجار   
4   5           2               1  31.4980  34.456       يوسف عادل زعرب   
5   6           4               3  31.5600  34.503      سامي رامي سلامة   
6   7           6               3  31.5580  34.499     فراس ناصر العمري   
7   8           1               3  31.5620  34.508     طارق جمال الدبعي   
8   9           3               3  31.5550  34.493      باسل حازم حمدان   
9  10           5               3  31.5520  34.488    نادر وليد الشوبكي   

                        email  
0  mohammad.abuomar@gmail.com  
1  ahmad.alhassan@hotmail.com  
2     omar.alsharif@gmail.com  
3     khalid.najjar@yahoo.co

In [5]:
with engine.connect() as conn:
    df_responses = pd.read_sql(text("""
        SELECT
            rr.id,
            rr.donor_id,
            rr.blood_request_id,
            rr.status,
            rr.created_at,
            rr.responded_at
        FROM request_responses rr
        LIMIT 10
    """), conn)

print(f"عدد الـ responses: {len(df_responses)}")
print(df_responses)

عدد الـ responses: 10
   id  donor_id  blood_request_id  status          created_at  \
0   1         1                 1       0 2026-03-19 07:22:12   
1   2         5                 1       1 2026-03-19 07:29:12   
2   3        21                 1       0 2026-03-19 07:36:12   
3   4         8                 1       4 2026-03-19 07:26:12   
4   5        11                 1       4 2026-03-19 07:49:12   
5   6         2                 2       0 2026-03-19 06:04:12   
6   7         6                 2       5 2026-03-19 05:54:12   
7   8        12                 2       1 2026-03-19 06:29:12   
8   9        20                 2       2 2026-03-19 06:44:12   
9  10        10                 3       3 2026-03-14 08:14:12   

         responded_at  
0 2026-03-19 07:22:12  
1 2026-03-19 07:29:12  
2 2026-03-19 07:36:12  
3 2026-03-19 07:26:12  
4 2026-03-19 07:49:12  
5 2026-03-19 06:04:12  
6 2026-03-19 05:54:12  
7 2026-03-19 06:29:12  
8 2026-03-19 06:44:12  
9 2026-03-14 08:14:12 

In [6]:
with engine.connect() as conn:
    stats = conn.execute(text("""
        SELECT
            (SELECT COUNT(*) FROM donors)           AS total_donors,
            (SELECT COUNT(*) FROM blood_requests)   AS total_requests,
            (SELECT COUNT(*) FROM request_responses) AS total_responses,
            (SELECT COUNT(*) FROM request_responses WHERE status = 1) AS accepted
    """)).fetchone()

print("=== إحصائيات قاعدة البيانات ===")
print(f"المتبرعين:        {stats.total_donors}")
print(f"طلبات الدم:       {stats.total_requests}")
print(f"الـ responses:    {stats.total_responses}")
print(f"المقبولة:         {stats.accepted}")

=== إحصائيات قاعدة البيانات ===
المتبرعين:        25
طلبات الدم:       8
الـ responses:    22
المقبولة:         5


In [7]:
with engine.connect() as conn:
    df_all = pd.read_sql(text("""
        SELECT
            rr.donor_id,
            rr.status,
            rr.created_at,
            rr.responded_at
        FROM request_responses rr
    """), conn)

# شوف كم response لكل status
status_map = {
    0: 'PENDING',
    1: 'ACCEPTED',
    2: 'DECLINED',
    3: 'COMPLETED',
    4: 'IGNORED',
    5: 'NO_SHOW'
}

df_all['status_name'] = df_all['status'].map(status_map)
print(df_all['status_name'].value_counts())

status_name
PENDING      5
ACCEPTED     5
IGNORED      3
COMPLETED    3
NO_SHOW      2
DECLINED     2
Name: count, dtype: int64


In [18]:
import numpy as np

with engine.connect() as conn:
    df_stats = pd.read_sql(text("""
        SELECT
            d.id as donor_id,
            COUNT(rr.id)                                      AS total_responses,
            COUNT(CASE WHEN rr.status = 1 THEN 1 END)         AS accepted_count,
            COUNT(CASE WHEN rr.status = 2 THEN 1 END)         AS declined_count,
            COUNT(CASE WHEN rr.status = 4 THEN 1 END)         AS ignored_count,
            DATEDIFF(NOW(), MAX(rr.responded_at))             AS days_since_last
        FROM donors d
        LEFT JOIN request_responses rr
               ON d.id = rr.donor_id
              AND rr.status IN (1, 2, 3, 4, 5)
        GROUP BY d.id
        ORDER BY d.id
    """), conn)

# احسب الـ features
df_stats['acceptance_rate'] = (
    df_stats['accepted_count'] /
    df_stats['total_responses'].clip(lower=1)
)

df_stats['recency_score'] = np.exp(
    -df_stats['days_since_last'].fillna(999) / 60
)

print(df_stats)

    donor_id  total_responses  accepted_count  declined_count  ignored_count  \
0          1                0               0               0              0   
1          2                0               0               0              0   
2          3                0               0               0              0   
3          4                1               1               0              0   
4          5                1               1               0              0   
5          6                1               0               0              0   
6          7                0               0               0              0   
7          8                1               0               0              1   
8          9                0               0               0              0   
9         10                1               0               0              0   
10        11                1               0               0              1   
11        12                1           

In [12]:
# rule-based score مبسط
df_stats['score'] = (
    (df_stats['acceptance_rate'] * 0.50) +
    (df_stats['recency_score']   * 0.30) +
    (df_stats['accepted_count'] / 10).clip(0, 1) * 0.20
).round(4)

# رتّب من الأعلى للأدنى
top_donors = df_stats.sort_values('score', ascending=False)

print("=== أفضل المتبرعين (Rule-Based Score) ===")
print(top_donors[['donor_id', 'total_responses', 'acceptance_rate', 'recency_score', 'score']].head(10))

=== أفضل المتبرعين (Rule-Based Score) ===
    donor_id  total_responses  acceptance_rate  recency_score   score
4          5                1              1.0       0.983471  0.8150
3          4                1              1.0       0.983471  0.8150
11        12                1              1.0       0.983471  0.8150
14        15                1              1.0       0.983471  0.8150
12        13                2              0.5       0.983471  0.5650
10        11                1              0.0       0.983471  0.2950
7          8                1              0.0       0.983471  0.2950
5          6                1              0.0       0.983471  0.2950
19        20                1              0.0       0.983471  0.2950
21        22                1              0.0       0.904837  0.2715


In [13]:
# فلتر — بس الـ responses المعروفة (مش PENDING)
df_training = df_stats[df_stats['total_responses'] > 0].copy()

# الـ label: قبل(1) أو ما قبل(0)
# بنعتبر ACCEPTED فقط = 1
df_training['label'] = (df_training['accepted_count'] > 0).astype(int)

print(f"عدد donors للتدريب: {len(df_training)}")
print(f"قبلوا (label=1):    {df_training['label'].sum()}")
print(f"ما قبلوا (label=0): {(df_training['label'] == 0).sum()}")
print()

# هل يكفي للتدريب؟
if len(df_training) < 50:
    print("⚠️ البيانات قليلة للـ XGBoost الحين")
    print("✅ بس نقدر نبني الـ Rule-Based Scorer كامل")
    print("✅ والـ XGBoost رح يشتغل تلقائياً لما تتراكم البيانات")

عدد donors للتدريب: 14
قبلوا (label=1):    5
ما قبلوا (label=0): 9

⚠️ البيانات قليلة للـ XGBoost الحين
✅ بس نقدر نبني الـ Rule-Based Scorer كامل
✅ والـ XGBoost رح يشتغل تلقائياً لما تتراكم البيانات


In [2]:
# Cell 13 — جرّب بـ threshold = 1 (response وحدة تكفي)

def calculate_rule_based_score_v2(row, min_history=1):
    total = row['total_responses']

    if total < min_history:
        return {
            'donor_id':      row['donor_id'],
            'score':         0.5,
            'is_cold_start': True,
            'source':        'cold_start'
        }

    acceptance_rate = row['accepted_count'] / total
    recency_score   = row['recency_score']
    loyalty_score   = min(row['accepted_count'] / 10, 1.0)

    score = round(
        (acceptance_rate * 0.50) +
        (recency_score   * 0.30) +
        (loyalty_score   * 0.20),
        4
    )

    return {
        'donor_id':      row['donor_id'],
        'score':         score,
        'is_cold_start': False,
        'source':        'rule_based'
    }

# min_history=1 للتطوير، في الإنتاج رح يكون 5
results_v2 = df_stats.apply(
    lambda row: calculate_rule_based_score_v2(row, min_history=1),
    axis=1
)
df_scores_v2 = pd.DataFrame(results_v2.tolist())

print("=== النتائج بـ min_history=1 ===")
print(df_scores_v2.sort_values('score', ascending=False).head(15))

NameError: name 'df_stats' is not defined

In [1]:
def epsilon_greedy_select(df_scores, budget=10, epsilon=0.20):
    """
    محاكاة لـ DonorScoringService::scoreAndSelect()
    budget  = عدد الإشعارات المسموح فيها
    epsilon = نسبة الـ exploration
    """
    # فصل cold-start عن المسجّلين
    cold_start  = df_scores[df_scores['is_cold_start'] == True]
    with_scores = df_scores[df_scores['is_cold_start'] == False].sort_values(
        'score', ascending=False
    )

    # حساب الـ slots
    exploit_slots = int(budget * (1 - epsilon))   # 8
    explore_slots = budget - exploit_slots          # 2

    # اختيار
    exploiters = with_scores.head(exploit_slots)
    explorers  = cold_start.sample(
        min(explore_slots, len(cold_start)),
        random_state=42
    )

    selected = pd.concat([exploiters, explorers])

    print(f"=== Epsilon-Greedy Selection ===")
    print(f"Budget:          {budget} إشعار")
    print(f"Epsilon:         {epsilon} ({epsilon*100:.0f}% exploration)")
    print(f"Exploit slots:   {exploit_slots}")
    print(f"Explore slots:   {explore_slots}")
    print(f"Cold-start pool: {len(cold_start)}")
    print(f"Scored pool:     {len(with_scores)}")
    print()
    print("المختارين للإشعار:")
    print(selected[['donor_id', 'score', 'is_cold_start', 'source']])

    return selected

selected_v2 = epsilon_greedy_select(df_scores_v2, budget=10, epsilon=0.20)


NameError: name 'df_scores_v2' is not defined